# K Sweep: encontrar K y modelo HMM optimo por dataset

Entrena caches HMM y corre experiments secuencialmente.
**Resumible**: si paras y vuelves, salta lo ya hecho (caches en disco, metrics en disco).

In [1]:
import sys, os, gc, time
import numpy as np
import pandas as pd
import torch

# Fijar CWD a la raiz del repo (independiente de donde se lance el NB)
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..')) if '__file__' in dir() else os.path.abspath('..')
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# ============================================================
# CONFIGURACION
# ============================================================

K_VALUES = [3, 4, 5, 6, 7, 8, 9, 10]
HMM_MODELS = ['hmm_soft', 'hmm_soft_residual']

DATASETS = {
    'ETTh1': {
        'root_path': './dataset/ETT-small/',
        'data_path': 'ETTh1.csv',
        'data': 'ETTh1',
        'target': 'OT',
    },
    'ETTh2': {
        'root_path': './dataset/ETT-small/',
        'data_path': 'ETTh2.csv',
        'data': 'ETTh2',
        'target': 'OT',
    },
    'Weather': {
        'root_path': './dataset/weather/',
        'data_path': 'weather.csv',
        'data': 'Weather',
        'target': 'OT',
    },
    'Electricity': {
        'root_path': './dataset/electricity/',
        'data_path': 'electricity.csv',
        'data': 'custom',
        'target': 'OT',
    },
}

# Transformer config (fija para todas las comparaciones)
TRANSFORMER_CFG = dict(
    d_model=64, n_heads=4, e_layers=2, d_ff=128,
    dropout=0.1, batch_size=32, learning_rate=0.001,
    lradj='cosine', train_epochs=30, patience=7,
    seq_len=96, pred_len=96, label_len=48,
)

CACHE_DIR = './cache'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'CWD: {os.getcwd()}')
print(f'Datasets: {list(DATASETS.keys())}')
print(f'K values: {K_VALUES}')
print(f'HMM models: {HMM_MODELS}')
print(f'Total experiments: {len(DATASETS) * len(K_VALUES) * len(HMM_MODELS)} HMM + skip si ya existen')

CWD: /home/jaime/TFG/RITMO
Datasets: ['ETTh1', 'ETTh2', 'Weather', 'Electricity']
K values: [3, 4, 5, 6, 7, 8, 9, 10]
HMM models: ['hmm_soft', 'hmm_soft_residual']
Total experiments: 64 HMM + skip si ya existen


## Celda 2 — Entrenar caches HMM
Para cada dataset x K, entrena Baum-Welch y guarda cache. Skip si ya existe.

In [2]:
from hmm.baum_welch import baum_welch
from sklearn.preprocessing import StandardScaler

def load_train_data(ds_cfg):
    """Carga train set normalizado per-window para Baum-Welch."""
    path = os.path.join(ds_cfg['root_path'], ds_cfg['data_path'])
    df = pd.read_csv(path)
    target = ds_cfg['target']
    
    # Reorder columns if custom (same logic as Dataset_Custom)
    if ds_cfg['data'] == 'custom':
        cols = list(df.columns)
        cols.remove(target)
        cols.remove('date')
        df = df[['date'] + cols + [target]]
    
    # Split: ETTh1/ETTh2 = 12 months, others = 70%
    if ds_cfg['data'] in ('ETTh1', 'ETTh2'):
        n_train = 12 * 30 * 24  # 8640
    else:
        n_train = int(len(df) * 0.7)
    
    train_vals = df[target].values[:n_train].astype(np.float64)
    
    # RevIN per-window normalization (same as HMM training pipeline)
    seq_len = 96
    windows = []
    for start in range(0, len(train_vals) - seq_len + 1, seq_len):
        w = train_vals[start:start + seq_len]
        m, s = w.mean(), max(w.std(), 1e-5)
        windows.append((w - m) / s)
    
    train_norm = np.concatenate(windows)
    return train_norm


total_caches = len(DATASETS) * len(K_VALUES)
done = 0

for ds_name, ds_cfg in DATASETS.items():
    # Cache uses args.data.lower() as key
    cache_key = ds_cfg['data'].lower()
    train_data = None  # lazy load
    
    for K in K_VALUES:
        cache_path = f'{CACHE_DIR}/hmm_{cache_key}_K{K}.pth'
        done += 1
        
        if os.path.exists(cache_path):
            c = torch.load(cache_path, weights_only=False)
            conv = c.get('converged', '?')
            n_iter = c.get('n_iter', '?')
            print(f'[{done}/{total_caches}] {ds_name} K={K}: SKIP (existe, converged={conv}, iter={n_iter})')
            continue
        
        # Lazy load train data (solo si necesitamos entrenar)
        if train_data is None:
            print(f'  Loading {ds_name} train data...')
            train_data = load_train_data(ds_cfg)
            print(f'  {len(train_data)} timesteps')
        
        print(f'[{done}/{total_caches}] {ds_name} K={K}: training...', end=' ', flush=True)
        t0 = time.time()
        result = baum_welch(train_data, K=K, max_iter=2000, epsilon=1e-4, random_state=42, verbose=False)
        elapsed = time.time() - t0
        
        cache = {
            'A': torch.from_numpy(result['A']).float(),
            'pi': torch.from_numpy(result['pi']).float(),
            'mu': torch.from_numpy(result['mu']).float(),
            'sigma': torch.from_numpy(result['sigma']).float(),
            'log_likelihood': result['log_likelihood'],
            'converged': result['converged'],
            'n_iter': result['n_iter'],
        }
        torch.save(cache, cache_path)
        print(f'done ({elapsed:.0f}s, iter={result["n_iter"]}, converged={result["converged"]})')
    
    # Liberar memoria entre datasets
    del train_data
    gc.collect()

print(f'\nTodos los caches listos.')

  Loading ETTh1 train data...
  8640 timesteps
[1/32] ETTh1 K=3: training... done (11s, iter=19, converged=True)
[2/32] ETTh1 K=4: training... done (21s, iter=37, converged=True)
[3/32] ETTh1 K=5: training... done (48s, iter=87, converged=True)
[4/32] ETTh1 K=6: training... done (55s, iter=100, converged=True)
[5/32] ETTh1 K=7: training... done (73s, iter=134, converged=True)
[6/32] ETTh1 K=8: training... done (144s, iter=259, converged=True)
[7/32] ETTh1 K=9: training... done (96s, iter=171, converged=True)
[8/32] ETTh1 K=10: training... done (204s, iter=328, converged=True)
  Loading ETTh2 train data...
  8640 timesteps
[9/32] ETTh2 K=3: training... done (29s, iter=53, converged=True)
[10/32] ETTh2 K=4: training... done (56s, iter=99, converged=True)
[11/32] ETTh2 K=5: training... done (88s, iter=156, converged=True)
[12/32] ETTh2 K=6: training... done (57s, iter=99, converged=True)
[13/32] ETTh2 K=7: training... done (92s, iter=157, converged=True)
[14/32] ETTh2 K=8: training... don

## Celda 3 — K sweep experiments
Para cada dataset x K x modelo HMM, entrena y evalua. Skip si metrics.npy ya existe.

In [3]:
import subprocess

def result_exists(ds_cfg, technique, K):
    """Comprueba si el resultado ya existe en results/."""
    des = f'ksweep_{technique}_K{K}'
    # Construir setting string (misma formula que run.py)
    setting = (
        f'plan_a_{ds_cfg["data"]}_96_96_TransformerCommon_{ds_cfg["data"]}'
        f'_ftS_sl96_ll48_pl96'
        f'_dm{TRANSFORMER_CFG["d_model"]}'
        f'_nh{TRANSFORMER_CFG["n_heads"]}'
        f'_el{TRANSFORMER_CFG["e_layers"]}'
        f'_dl1'
        f'_df{TRANSFORMER_CFG["d_ff"]}'
        f'_expand2_dc4_fc1_ebtimeF_dtTrue'
        f'_{des}_0'
    )
    metrics_path = f'./results/{setting}/metrics.npy'
    return os.path.exists(metrics_path), metrics_path


def run_experiment(ds_name, ds_cfg, technique, K):
    """Corre un experiment via subprocess (aislado en memoria)."""
    des = f'ksweep_{technique}_K{K}'
    cmd = [
        'python', '-u', 'run.py',
        '--task_name', 'plan_a',
        '--is_training', '1',
        '--root_path', ds_cfg['root_path'],
        '--data_path', ds_cfg['data_path'],
        '--model_id', f'{ds_cfg["data"]}_96_96',
        '--model', 'TransformerCommon',
        '--data', ds_cfg['data'],
        '--features', 'S',
        '--target', ds_cfg['target'],
        '--seq_len', str(TRANSFORMER_CFG['seq_len']),
        '--label_len', str(TRANSFORMER_CFG['label_len']),
        '--pred_len', str(TRANSFORMER_CFG['pred_len']),
        '--enc_in', '1',
        '--dec_in', '1',
        '--c_out', '1',
        '--d_model', str(TRANSFORMER_CFG['d_model']),
        '--n_heads', str(TRANSFORMER_CFG['n_heads']),
        '--e_layers', str(TRANSFORMER_CFG['e_layers']),
        '--d_ff', str(TRANSFORMER_CFG['d_ff']),
        '--dropout', str(TRANSFORMER_CFG['dropout']),
        '--batch_size', str(TRANSFORMER_CFG['batch_size']),
        '--learning_rate', str(TRANSFORMER_CFG['learning_rate']),
        '--lradj', TRANSFORMER_CFG['lradj'],
        '--train_epochs', str(TRANSFORMER_CFG['train_epochs']),
        '--patience', str(TRANSFORMER_CFG['patience']),
        '--use_gpu', '0',
        '--technique', technique,
        '--hmm_k', str(K),
        '--des', des,
        '--itr', '1',
    ]
    
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    elapsed = time.time() - t0
    
    # Extraer MSE/MAE del output
    mse_line = [l for l in proc.stdout.split('\n') if l.startswith('mse:')]
    if mse_line:
        return elapsed, mse_line[-1]
    else:
        # Error
        err = proc.stderr[-500:] if proc.stderr else 'no stderr'
        return elapsed, f'ERROR: {err}'


total_exp = len(DATASETS) * len(K_VALUES) * len(HMM_MODELS)
done = 0

for ds_name, ds_cfg in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'  {ds_name}')
    print(f'{"="*60}')
    
    for K in K_VALUES:
        for technique in HMM_MODELS:
            done += 1
            exists, mpath = result_exists(ds_cfg, technique, K)
            
            if exists:
                m = np.load(mpath)
                print(f'[{done}/{total_exp}] {ds_name} {technique} K={K}: SKIP (MSE={m[1]:.6f})')
                continue
            
            print(f'[{done}/{total_exp}] {ds_name} {technique} K={K}: running...', end=' ', flush=True)
            elapsed, result_str = run_experiment(ds_name, ds_cfg, technique, K)
            print(f'({elapsed:.0f}s) {result_str}')
    
    # Limpiar checkpoints de este dataset para liberar disco
    import glob
    for cp in glob.glob(f'./checkpoints/plan_a_{ds_cfg["data"]}_*_ksweep_*'):
        import shutil
        shutil.rmtree(cp, ignore_errors=True)
    gc.collect()

print(f'\nK sweep completo.')


  ETTh1
[1/64] ETTh1 hmm_soft K=3: running... (284s) mse:0.05999751389026642, mae:0.18903757631778717
[2/64] ETTh1 hmm_soft_residual K=3: running... (290s) mse:0.062324099242687225, mae:0.19061806797981262
[3/64] ETTh1 hmm_soft K=4: running... (340s) mse:0.05866996943950653, mae:0.1882050782442093
[4/64] ETTh1 hmm_soft_residual K=4: running... (313s) mse:0.057917624711990356, mae:0.18402670323848724
[5/64] ETTh1 hmm_soft K=5: running... (348s) mse:0.05944887548685074, mae:0.1881420910358429
[6/64] ETTh1 hmm_soft_residual K=5: running... (476s) mse:0.0660511925816536, mae:0.19870653748512268
[7/64] ETTh1 hmm_soft K=6: running... (411s) mse:0.060600657016038895, mae:0.18947947025299072
[8/64] ETTh1 hmm_soft_residual K=6: running... (427s) mse:0.05895479768514633, mae:0.18549545109272003
[9/64] ETTh1 hmm_soft K=7: running... (477s) mse:0.06167454645037651, mae:0.18965250253677368
[10/64] ETTh1 hmm_soft_residual K=7: running... (474s) mse:0.06098835915327072, mae:0.19077931344509125
[11/6

KeyboardInterrupt: 

## Celda 4 — Tabla resumen
Lee todos los resultados y muestra ranking por dataset.

In [ ]:
import re
from collections import defaultdict

results_dir = './results'
rows = []

for d in sorted(os.listdir(results_dir)):
    mpath = os.path.join(results_dir, d, 'metrics.npy')
    if not os.path.exists(mpath) or 'ksweep' not in d:
        continue
    m = np.load(mpath)
    mae, mse = float(m[0]), float(m[1])
    
    # Parse dataset
    for ds in ['ETTh1', 'ETTh2', 'Weather', 'custom']:
        if f'_{ds}_' in d or d.startswith(f'plan_a_{ds}'):
            dataset = 'Electricity' if ds == 'custom' else ds
            break
    else:
        continue
    
    # Parse technique and K
    tech_match = re.search(r'ksweep_(hmm_soft(?:_residual)?)_K(\d+)', d)
    if tech_match:
        technique = tech_match.group(1)
        K = int(tech_match.group(2))
    else:
        continue
    
    rows.append({'dataset': dataset, 'technique': technique, 'K': K, 'MSE': mse, 'MAE': mae})

df = pd.DataFrame(rows)
if len(df) == 0:
    print('No hay resultados ksweep todavia.')
else:
    for ds in ['ETTh1', 'ETTh2', 'Weather', 'Electricity']:
        sub = df[df['dataset'] == ds].sort_values('MSE')
        if len(sub) == 0:
            continue
        best_mse = sub['MSE'].min()
        print(f'\n{"="*65}')
        print(f'  {ds}')
        print(f'{"="*65}')
        print(f'{"Rank":>4} {"Technique":>20} {"K":>3} {"MSE":>10} {"MAE":>10} {"Gap":>7}')
        print(f'  {"-"*58}')
        for rank, (_, row) in enumerate(sub.iterrows(), 1):
            gap = (row['MSE'] - best_mse) / best_mse * 100
            marker = ' ***' if rank == 1 else ''
            print(f'{rank:4d} {row["technique"]:>20} {row["K"]:3d} {row["MSE"]:10.6f} {row["MAE"]:10.6f} {gap:+6.1f}%{marker}')
    
    # Resumen: mejor K por dataset
    print(f'\n{"="*65}')
    print(f'  RESUMEN: Mejor K y modelo por dataset')
    print(f'{"="*65}')
    for ds in ['ETTh1', 'ETTh2', 'Weather', 'Electricity']:
        sub = df[df['dataset'] == ds].sort_values('MSE')
        if len(sub) == 0:
            continue
        best = sub.iloc[0]
        print(f'  {ds:15s} -> {best["technique"]} K={best["K"]:.0f}  MSE={best["MSE"]:.6f}')